In [7]:

import numpy as np


def frame_energy_db(x, fs, frame_ms=20, hop_ms=10):
    frame_len = int(fs * frame_ms / 1000)
    hop_len = int(fs * hop_ms / 1000)

    frame_len = max(1, frame_len)
    hop_len = max(1, hop_len)

    # Them zero neu khung cuoi chua du mau
    n_frames = max(
        1, int(np.ceil(max(0, len(x) - frame_len) / hop_len)) + 1
    )
    total_len = (n_frames - 1) * hop_len + frame_len
    padded = np.pad(x, (0, max(0, total_len - len(x))))

    frames = np.array([
        padded[i * hop_len:i * hop_len + frame_len]
        for i in range(n_frames)
    ])

    energy = np.mean(frames ** 2, axis=1)
    energy_db = 10 * np.log10(energy + 1e-12)

    return energy_db, frame_len, hop_len


def noise_gate_fullband(
    x, fs, threshold_dB, hold_ms, attack_ms, release_ms
):
    x = np.asarray(x, dtype=float)

    if x.ndim != 1:
        raise ValueError("Tin hieu dau vao phai la mono")

    if len(x) == 0:
        raise ValueError("Tin hieu rong")

    energy_db, frame_len, hop_len = frame_energy_db(x, fs)
    n_frames = len(energy_db)

    # Cho phep threshold la mot so hoac mot mang theo tung khung
    threshold = np.asarray(threshold_dB, dtype=float)

    if threshold.ndim == 0:
        threshold = np.full(n_frames, float(threshold))
    elif len(threshold) != n_frames:
        raise ValueError("So nguong phai bang so khung tin hieu")

    # Hysteresis: nguong dong thap hon nguong mo 3 dB
    close_threshold = threshold - 3.0

    hop_ms = hop_len * 1000 / fs
    hold_frames = int(round(hold_ms / hop_ms))

    frame_gain = np.zeros(n_frames)
    is_open = False
    hold_counter = 0

    for m in range(n_frames):
        if not is_open:
            if energy_db[m] > threshold[m]:
                is_open = True
                hold_counter = hold_frames
        else:
            if energy_db[m] > close_threshold[m]:
                hold_counter = hold_frames
            elif hold_counter > 0:
                hold_counter -= 1
            else:
                is_open = False

        frame_gain[m] = 1.0 if is_open else 0.0

    # Noi suy Gain tu muc khung sang muc tung mau
    frame_times = np.arange(n_frames) * hop_len
    sample_times = np.arange(len(x))

    gain = np.interp(
        sample_times,
        frame_times,
        frame_gain,
        left=frame_gain[0],
        right=frame_gain[-1]
    )

    # Lam muot Attack va Release
    attack_coeff = (
        np.exp(-1 / (fs * attack_ms / 1000))
        if attack_ms > 0 else 0.0
    )
    release_coeff = (
        np.exp(-1 / (fs * release_ms / 1000))
        if release_ms > 0 else 0.0
    )

    if attack_ms > 0 or release_ms > 0:
        smooth_gain = np.zeros_like(gain)
        smooth_gain[0] = gain[0]

        for n in range(1, len(gain)):
            if gain[n] > smooth_gain[n - 1]:
                alpha = attack_coeff
            else:
                alpha = release_coeff

            smooth_gain[n] = (
                alpha * smooth_gain[n - 1]
                + (1 - alpha) * gain[n]
            )

        gain = smooth_gain

    y = x * gain

    return y, gain


In [8]:
%pip install soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

# Doc file am thanh
x, fs = sf.read("speech_with_noise.wav")

# Neu file stereo, chuyen thanh mono
if x.ndim == 2:
    x = np.mean(x, axis=1)

# Thong tin tin hieu
duration = len(x) / fs

print("Tan so lay mau:", fs, "Hz")
print("So mau:", len(x))
print("Thoi luong:", duration, "giay")
print("Bien do lon nhat:", np.max(np.abs(x)))

# Ve dang song dau vao
t = np.arange(len(x)) / fs

plt.figure(figsize=(12, 4))
plt.plot(t, x, linewidth=0.5)
plt.xlabel("Thoi gian (s)")
plt.ylabel("Bien do")
plt.title("Tin hieu tieng noi co nhieu")
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'x' is not defined

In [ ]:
#Manual
def threshold_manual(x, fs, noise_ms=500, margin_db=5):
    n = min(len(x), int(fs * noise_ms / 1000))
    noise = x[:max(1, n)]

    rms = np.sqrt(np.mean(noise ** 2) + 1e-12)
    noise_db = 20 * np.log10(rms)

    threshold = noise_db + margin_db

    return threshold

th_manual = threshold_manual(x, fs)
print("Manual threshold:", th_manual, "dBFS")

#Static Automatic
def threshold_static(x, fs, noise_ms=300, k=3):
    energy_db, _, _ = frame_energy_db(x, fs)

    n_frames = max(1, int(np.ceil(noise_ms / 10)))
    noise_frames = energy_db[:min(n_frames, len(energy_db))]

    mu_noise = np.mean(noise_frames)
    sigma_noise = np.std(noise_frames)

    threshold = mu_noise + k * sigma_noise

    return threshold

th_static = threshold_static(x, fs)
print("Static threshold:", th_static, "dBFS")

#Percentile

def threshold_percentile(x, fs, percentile=15, margin_db=5):
    energy_db, _, _ = frame_energy_db(x, fs)

    noise_level = np.percentile(energy_db, percentile)
    threshold = noise_level + margin_db

    return threshold

th_percentile = threshold_percentile(x, fs)
print("Percentile threshold:", th_percentile, "dBFS")

#Minimum Statistics

from scipy.ndimage import minimum_filter1d


def threshold_minimum_statistics(
    x, fs, window_s=1.0, margin_db=5
):
    energy_db, _, _ = frame_energy_db(x, fs)

    hop_ms = 10
    window_frames = max(1, int(window_s * 1000 / hop_ms))

    if window_frames % 2 == 0:
        window_frames += 1

    noise_estimate = minimum_filter1d(
        energy_db,
        size=window_frames,
        mode="nearest"
    )

    threshold = noise_estimate + margin_db

    return threshold
th_minimum = threshold_minimum_statistics(x, fs)

print("Minimum Statistics:")
print("Threshold min:", np.min(th_minimum), "dBFS")
print("Threshold max:", np.max(th_minimum), "dBFS")


#so sánh 4 ngưỡng

thresholds = {
    "Manual": th_manual,
    "Static Automatic": th_static,
    "Percentile": th_percentile,
    "Minimum Statistics": th_minimum
}

for name, value in thresholds.items():
    if np.ndim(value) == 0:
        print(f"{name}: {value:.2f} dBFS")
    else:
        print(
            f"{name}: {np.min(value):.2f} "
            f"to {np.max(value):.2f} dBFS"
        )

energy_db, _, _ = frame_energy_db(x, fs)
frame_time = np.arange(len(energy_db)) * 0.01

plt.figure(figsize=(12, 5))
plt.plot(frame_time, energy_db, label="Frame energy")

for name, value in thresholds.items():
    if np.ndim(value) == 0:
        plt.axhline(value, linestyle="--", label=name)
    else:
        plt.plot(frame_time, value, label=name)

plt.xlabel("Thoi gian (s)")
plt.ylabel("Muc nang luong (dBFS)")
plt.title("So sanh bon phuong phap tinh nguong")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()





In [ ]:
#Cấu hình 1 — Gate nhị phân thô
y1, g1 = noise_gate_fullband(
    x, fs,
    threshold_dB=th_percentile,
    hold_ms=0,
    attack_ms=0,
    release_ms=0
)

sf.write("gate_binary.wav", y1, fs)
print("Da luu gate_binary.wav")

#Gate có Hysteresis và Hold
y2, g2 = noise_gate_fullband(
    x, fs,
    threshold_dB=th_percentile,
    hold_ms=200,
    attack_ms=0,
    release_ms=0
)

sf.write("gate_hysteresis.wav", y2, fs)
print("Da luu gate_hysteresis.wav")
#Gate hoàn chỉnh

y3, g3 = noise_gate_fullband(
    x, fs,
    threshold_dB=th_percentile,
    hold_ms=200,
    attack_ms=10,
    release_ms=100
)

sf.write("gate_complete.wav", y3, fs)
print("Da luu gate_complete.wav")



In [ ]:
#Đối sánh dạng sóng

t = np.arange(len(x)) / fs

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(t, x, linewidth=0.5)
axes[0].set_title("Tin hieu dau vao co nhieu")

axes[1].plot(t, y1, linewidth=0.5)
axes[1].set_title("Gate nhi phan")

axes[2].plot(t, y2, linewidth=0.5)
axes[2].set_title("Gate Hysteresis + Hold")

axes[3].plot(t, y3, linewidth=0.5)
axes[3].set_title("Gate Hysteresis + Attack/Release")

for ax in axes:
    ax.set_ylabel("Amplitude")
    ax.grid(True)

axes[-1].set_xlabel("Thoi gian (s)")
plt.tight_layout()
plt.savefig("waveform_comparison.png", dpi=200)
plt.show()
#Đối sánh đường bao hệ số \(g[n]\)


plt.figure(figsize=(14, 5))

plt.plot(t, g1, label="Binary", linewidth=0.8)
plt.plot(t, g2, label="Hysteresis + Hold", linewidth=0.8)
plt.plot(t, g3, label="Complete Gate", linewidth=0.8)

plt.xlabel("Thoi gian (s)")
plt.ylabel("Gain g[n]")
plt.title("So sanh duong bao he so Gain")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("gain_comparison.png", dpi=200)
plt.show()

#ghe ba file đầu ra
import sounddevice as sd

print("Dang phat Gate nhi phan...")
sd.play(y1, fs)
sd.wait()

print("Dang phat Gate Hysteresis...")
sd.play(y2, fs)
sd.wait()

print("Dang phat Gate hoan chinh...")
sd.play(y3, fs)
sd.wait()
